# 🪙 반사실적 설명

반사실적 설명은 원하는 출력을 얻기 위해 필요한 입력을 보는 데 좋습니다.  
우리의 경우, 터키에서 노래를 인기 있게 만드는 데 필요한 입력을 확인하고 싶을 수 있습니다.  
우리는 TrustyAI를 사용하여 이를 정확하게 테스트하고 얼마나 많은 변화가 필요한지 확인할 것입니다.

In [ ]:
!pip -q install "onnx" "onnxruntime" "numpy==1.26.4"

In [2]:
import pickle
import pandas as pd
import numpy as np
import onnxruntime as rt

In [ ]:
import warnings

# UserWarning 무시
warnings.filterwarnings("ignore", category=UserWarning)

먼저 우리가 노래를 인기 있게 만들고 싶은 국가를 선택합니다.  
또한 우리의 노래가 그 국가에서 인기가 있을 좋은 기회가 있다고 말하기 전에 봐야 할 확률을 선택합니다.  

In [ ]:
PRED_COUNTRY = "TR"
POPULAR_THRESHOLD = 0.3

그 다음 우리는 모델과 사전 및 사후 처리 아티팩트를 로드합니다.  

In [ ]:
onnx_session = rt.InferenceSession("convert-keras-to-onnx/onnx_model.onnx", providers=rt.get_available_providers())
onnx_input_name = onnx_session.get_inputs()[0].name
onnx_output_name = onnx_session.get_outputs()[0].name

with open('preprocess-data/scaler.pkl', 'rb') as handle:
    scaler = pickle.load(handle)

with open('preprocess-data/label_encoder.pkl', 'rb') as handle:
    label_encoder = pickle.load(handle)

### 데이터

그 다음 우리가 인기 있게 만들고 싶은 노래를 선택합니다.  
우리는 또한 노래 속성을 조금 처리할 것이고, 예를 들어 확장(scaling)하는 것처럼, 모델을 학습할 때 수행한 것과 동일하게 합니다. 이는 모델이 이해하는 입력을 가지고 있는지 확인하기 위한 것입니다. 

In [ ]:
song_properties = pd.read_parquet('../99-data_prep/song_properties.parquet')
favorite_song = song_properties.loc[song_properties["name"]=="Not Like Us"]
favorite_song

In [ ]:
song_properties = favorite_song[['is_explicit', 'duration_ms', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']]
song_properties.T

In [ ]:
scaled_feature = scaler.transform(song_properties)[0]
feature_values = {
    "is_explicit": scaled_feature[0],
    "duration_ms": scaled_feature[1],
    "danceability": scaled_feature[2],
    "energy": scaled_feature[3],
    "key": scaled_feature[4],
    "loudness": scaled_feature[5],
    "mode": scaled_feature[6],
    "speechiness": scaled_feature[7],
    "acousticness": scaled_feature[8],
    "instrumentalness": scaled_feature[9],
    "liveness": scaled_feature[10],
    "valence": scaled_feature[11],
    "tempo": scaled_feature[12]
}

feature_df = pd.DataFrame([feature_values])
feature_df.T

우리는 또한 모든 출력 이름을 어떻게 호출해야 하는지 설정합니다. 이는 국가 코드와 동일합니다.

In [ ]:
output_names = label_encoder.classes_
output_names

### 반사실적 분석

이제 모든 것이 설정되었으므로 반사실적 분석을 설정할 것입니다.  
여기서 먼저 예측 함수를 생성해야 합니다(모델이 기본적으로 pandas dataframe을 입력하고 출력하는 경우 필요하지 않음).  
그 다음 우리는 TrustyAI "Model"을 생성할 것이며, 이는 단순히 우리의 모델을 래핑하고 TrustyAI가 다양한 입력값에 대해 반복하는 데 사용할 것입니다.  
마지막으로 우리는 입력 각각에 대해 TrustyAI "domains"을 정의할 것입니다. 이는 TrustyAI에 입력이 무엇 사이의 값이 될 수 있는지 알려줍니다.

In [ ]:
def pred(x):
    x = x[0]
    x_dict = {name: np.asarray([[x[i]]]).astype(np.float32) for i, name in enumerate(feature_df.columns)}
    pred = onnx_session.run([onnx_output_name], x_dict)
    pred = np.squeeze(pred)
    pred = {output_names[i]: pred[i] for i in range(pred.shape[0])}
    print(f"예측 확률: {pred[PRED_COUNTRY]}")
    if pred[PRED_COUNTRY] >= POPULAR_THRESHOLD:
        pred = {PRED_COUNTRY: True}
    else:
        pred = {PRED_COUNTRY: False}
    return pd.DataFrame([pred])

In [ ]:
pred(feature_df.to_numpy())

In [12]:
from trustyai.model import Model

model = Model(pred, output_names=[PRED_COUNTRY])

In [ ]:
from trustyai.model.domain import feature_domain
_domains = {
        "is_explicit": (0.0, 1.0),
        "duration_ms": (0.0, 1.0),
        "danceability": (0.0, 1.0),
        "energy": (0.0, 1.0),
        "key": (0.0, 1.0),
        "loudness": (0.0, 1.0),
        "mode": (0.0, 1.0),
        "speechiness": (0.0, 1.0),
        "acousticness": (0.0, 1.0),
        "instrumentalness": (0.0, 1.0),
        "liveness": (0.0, 1.0),
        "valence": (0.0, 1.0),
        "tempo": (0.0, 1.0)
}
domains = {key: None for key  in feature_values.keys()}

for key in  _domains.keys():
        domains[key] = feature_domain(_domains[key])

domains = list(domains.values())

In [14]:
from trustyai.model import output
goal = [output(name=PRED_COUNTRY, dtype="bool", value=True)]

모델, 도메인 및 목표가 있으면 가능한 입력을 통해 실행하여 우리가 원하는 출력을 제공할 수 있는 입력을 볼 수 있습니다.  

In [ ]:
from trustyai.explainers import CounterfactualExplainer

STEPS=50
explainer = CounterfactualExplainer(steps=STEPS)
explanation = explainer.explain(inputs=feature_df, goal=goal, model=model, feature_domains=domains)

In [ ]:
model(explanation.proposed_features_dataframe.to_numpy())

실행이 완료되면 원래 입력(시작 부분에서 선택한 노래)에서 우리의 노래가 우리 국가에서 인기 있게 되도록 얼마나 많은 변화가 필요한지 볼 수 있습니다.  

In [ ]:
explanation.as_dataframe()

In [ ]:
df = explanation.as_dataframe()
df[df.difference != 0.0]

In [ ]:
if not df[df.difference != 0.0].empty:
    explanation.plot()
else:
    print(f"{PRED_COUNTRY} 국가에서 {STEPS} 스텝 내에 확률 {POPULAR_THRESHOLD}에 도달하지 못했습니다")